<a href="https://colab.research.google.com/github/phong6786789/OMNIVOICE-MOD/blob/main/colab_adam.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phong Subi - VOICEOMNI MOD

> **GPU:** T4 16GB (tự động) · **Thời gian:** ~3 phút lần đầu, ~30s/lần sau

## Hướng dẫn

1. **Runtime** → **Run all** (Ctrl+F9)
2. Nhập văn bản → Click **Tạo giọng nói**
3. Nghe + tải file WAV về

> Powered by OmniVoice (MIT License) — github.com/k2-fsa/OmniVoice


In [17]:
print('Dang cai dat (~1 phut)...')
!pip install -q omnivoice gradio "numpy<2.1" "requests==2.32.4"
print('Cai dat hoan tat!')


Dang cai dat (~1 phut)...
Cai dat hoan tat!


In [ ]:
print("🚀 Đang khởi động Phong Subi - OMNIVOICE MOD...")

# ============================================================
# IMPORT
# ============================================================

import os
import re
import time
import shutil
import logging
import requests
import numpy as np
import torch
import gradio as gr

from IPython.display import clear_output


# ============================================================
# PATCH TORCH
# ============================================================

import torch as _torch

if not hasattr(_torch, "_utils"):
    _torch._utils = _torch._C._utils


# ============================================================
# PATCH TRANSFORMERS
# ============================================================

import transformers as _tf


class _SafeAutoFeatureExtractor:

    @staticmethod
    def from_pretrained(model_name, **kwargs):

        try:

            from transformers import AutoConfig

            cfg = AutoConfig.from_pretrained(
                model_name,
                trust_remote_code=True,
                **kwargs
            )

            sr = getattr(
                cfg,
                "sampling_rate",
                24000
            )

        except Exception:

            sr = 24000


        class _Result:
            sampling_rate = sr


        return _Result()


_tf.AutoFeatureExtractor = _SafeAutoFeatureExtractor


# ============================================================
# OMNIVOICE
# ============================================================

from omnivoice import (
    OmniVoice,
    OmniVoiceGenerationConfig
)

from omnivoice.utils.common import get_best_device


# ============================================================
# LOGGING
# ============================================================

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s: %(message)s"
)

logger = logging.getLogger(__name__)


# ============================================================
# THƯ MỤC
# ============================================================

VOICE_DIR = "/content/voices"
CUSTOM_VOICE_DIR = "/content/custom_voices"

os.makedirs(
    VOICE_DIR,
    exist_ok=True
)

os.makedirs(
    CUSTOM_VOICE_DIR,
    exist_ok=True
)


# ============================================================
# GITHUB VOICES
# ============================================================

GITHUB_VOICE_BASE = (
    "https://raw.githubusercontent.com/"
    "phong6786789/"
    "OMNIVOICE-MOD/"
    "main/voices/"
)


# ============================================================
# 27 GIỌNG PRESET
# ============================================================

VOICE_DATA = {

    # ========================================================
    # NỮ - 12
    # ========================================================

    "♀ Khánh Huyền":
        "khanhhuyentvc_sample.mp3",

    "♀ Thùy Linh":
        "thuylinh_thuyetminh.mp3",

    "♀ Hoài An":
        "vi_female_hoaian_mb.mp3",

    "♀ Hồng Hạnh":
        "vi_female_honghanh_mn_podcast.mp3",

    "♀ Hồng Ngân":
        "vi_female_hongngan_mn_buon.mp3",

    "♀ Khánh Linh":
        "vi_female_khanhlinh_mb.mp3",

    "♀ Kim Phương":
        "vi_female_kimphuong_mb_tKhLy5k.mp3",

    "♀ Nova":
        "vi_female_nova_default.mp3",

    "♀ Thủy Tiên":
        "vi_female_thuytien_mn.mp3",

    "♀ Thùy Trang":
        "vi_female_thuytrang_mb_rzuuQ7F.mp3",

    "♀ Trâm Anh":
        "vi_female_tramanh_mb_sample.mp3",

    "♀ Trần Anh":
        "vi_female_trananh_mb.mp3",


    # ========================================================
    # NAM - 15
    # ========================================================

    "♂ Đức Trọng":
        "ductrong_sample2.mp3",

    "♂ Đăng Khoa":
        "vi_male_dangkhoa_mb.mp3",

    "♂ Echo":
        "vi_male_echo_default.mp3",

    "♂ Lê Đức":
        "vi_male_leduc_mb_FZBJAUZ.mp3",

    "♂ Lê Hoàng":
        "vi_male_lehoang_mb_SRojRwi.mp3",

    "♂ Lê Nghĩa":
        "vi_male_lenghia_mb_BITWyO7.mp3",

    "♂ Minh Quân":
        "vi_male_minhquan_mb.mp3",

    "♂ Minh Triết":
        "vi_male_minhtriet_mb.mp3",

    "♂ Onyx":
        "vi_male_onyx_default.mp3",

    "♂ Thành Trung":
        "vi_male_thanhtrung_mn_k8K8ONG.mp3",

    "♂ Trí Dũng":
        "vi_male_tridung_mn_sample.mp3",

    "♂ Tuấn Kiệt":
        "vi_male_tuankiet_mn.mp3",

    "♂ Văn Đức":
        "vi_male_vanduc_mn.mp3",

    "♂ Văn Duy":
        "vi_male_vanduy_mb.mp3",

    "♂ Adam":
        "samples_adam.mp3",
}


TOTAL_PRESET_VOICES = len(
    VOICE_DATA
)


# ============================================================
# DOWNLOAD FILE
# ============================================================

def download_file(
    url,
    destination
):

    response = requests.get(
        url,
        timeout=60
    )

    response.raise_for_status()

    with open(
        destination,
        "wb"
    ) as f:

        f.write(
            response.content
        )


# ============================================================
# TẢI 27 GIỌNG
# ============================================================

VOICE_FILES = {}

available_voices = []

failed_voices = []


print("\n========================================")

print(
    f"⬇️ KIỂM TRA / TẢI "
    f"{TOTAL_PRESET_VOICES} GIỌNG"
)

print("========================================")


for voice_name, filename in VOICE_DATA.items():

    local_path = os.path.join(
        VOICE_DIR,
        filename
    )

    raw_url = (
        GITHUB_VOICE_BASE
        +
        filename
    )


    if os.path.exists(
        local_path
    ):

        print(
            f"✅ {voice_name}"
        )

        VOICE_FILES[
            voice_name
        ] = local_path

        available_voices.append(
            voice_name
        )

        continue


    try:

        print(
            f"⬇️ {voice_name}"
        )


        download_file(
            raw_url,
            local_path
        )


        if os.path.getsize(
            local_path
        ) < 1000:

            os.remove(
                local_path
            )

            raise RuntimeError(
                "File tải về không hợp lệ."
            )


        VOICE_FILES[
            voice_name
        ] = local_path


        available_voices.append(
            voice_name
        )


        print(
            "   ✅ Hoàn tất"
        )


    except Exception as e:

        print(
            f"   ❌ {e}"
        )

        failed_voices.append(
            voice_name
        )


print("\n========================================")

print(
    f"✅ {len(available_voices)}/"
    f"{TOTAL_PRESET_VOICES} giọng sẵn sàng"
)

print("========================================\n")


# ============================================================
# GPU
# ============================================================

print(
    "🔍 Kiểm tra GPU..."
)


for i in range(30):

    if torch.cuda.is_available():

        print(
            "✅ GPU:",
            torch.cuda.get_device_name(0)
        )

        break

    time.sleep(1)

else:

    print(
        "⚠️ Không phát hiện GPU.\n"
        "Runtime → Change runtime type → T4 GPU"
    )


# ============================================================
# LOAD OMNIVOICE
# ============================================================

DEVICE = get_best_device()


MODEL_DTYPE = (

    torch.float16

    if torch.cuda.is_available()

    else torch.float32
)


logger.info(
    f"Loading OmniVoice on {DEVICE}..."
)


model = OmniVoice.from_pretrained(

    "k2-fsa/OmniVoice",

    device_map=DEVICE,

    dtype=MODEL_DTYPE,

    load_asr=True
)


SAMPLING_RATE = (
    model.sampling_rate
)


logger.info(
    f"✅ OmniVoice ready — "
    f"{SAMPLING_RATE} Hz"
)


# ============================================================
# GENERATION CONFIG
# ============================================================

GEN_CFG = OmniVoiceGenerationConfig(

    num_step=32,

    guidance_scale=1.8,

    denoise=True,

    preprocess_prompt=True,

    postprocess_output=True,

    position_temperature=5.0,

    class_temperature=0.2,

    pad_duration=0.1,

    fade_duration=0.1,
)


# ============================================================
# CACHE
# ============================================================

VOICE_PROMPT_CACHE = {}

CUSTOM_VOICE_FILES = {}

CUSTOM_VOICE_PROMPTS = {}

CUSTOM_VOICE_COUNTER = 0


# ============================================================
# DANH SÁCH GIỌNG
# ============================================================

def get_all_voice_names():

    return (
        list(
            VOICE_DATA.keys()
        )
        +
        list(
            CUSTOM_VOICE_FILES.keys()
        )
    )


# ============================================================
# FILE GIỌNG
# ============================================================

def get_voice_file(
    voice_name
):

    # ========================================================
    # CUSTOM
    # ========================================================

    if voice_name in CUSTOM_VOICE_FILES:

        return CUSTOM_VOICE_FILES[
            voice_name
        ]


    # ========================================================
    # PRESET
    # ========================================================

    if voice_name not in VOICE_DATA:

        return None


    filename = VOICE_DATA[
        voice_name
    ]


    local_path = os.path.join(
        VOICE_DIR,
        filename
    )


    # ========================================================
    # ĐÃ CÓ
    # ========================================================

    if os.path.exists(
        local_path
    ):

        VOICE_FILES[
            voice_name
        ] = local_path

        return local_path


    # ========================================================
    # CHƯA CÓ -> DOWNLOAD
    # ========================================================

    raw_url = (
        GITHUB_VOICE_BASE
        +
        filename
    )


    try:

        download_file(
            raw_url,
            local_path
        )


        if os.path.getsize(
            local_path
        ) < 1000:

            os.remove(
                local_path
            )

            raise RuntimeError(
                "File voice không hợp lệ."
            )


        VOICE_FILES[
            voice_name
        ] = local_path


        return local_path


    except Exception as e:

        if os.path.exists(
            local_path
        ):

            os.remove(
                local_path
            )


        raise gr.Error(
            f"Không tải được giọng "
            f"{voice_name}:\n{e}"
        )


# ============================================================
# VOICE PROMPT
# ============================================================

def get_voice_prompt(
    voice_name
):

    # ========================================================
    # CUSTOM CACHE
    # ========================================================

    if voice_name in CUSTOM_VOICE_PROMPTS:

        return CUSTOM_VOICE_PROMPTS[
            voice_name
        ]


    # ========================================================
    # PRESET CACHE
    # ========================================================

    if voice_name in VOICE_PROMPT_CACHE:

        return VOICE_PROMPT_CACHE[
            voice_name
        ]


    voice_file = get_voice_file(
        voice_name
    )


    if not voice_file:

        raise ValueError(
            "Không tìm thấy giọng."
        )


    logger.info(
        f"Creating VoiceClonePrompt: "
        f"{voice_name}"
    )


    voice_prompt = (
        model.create_voice_clone_prompt(
            ref_audio=voice_file
        )
    )


    if voice_name in CUSTOM_VOICE_FILES:

        CUSTOM_VOICE_PROMPTS[
            voice_name
        ] = voice_prompt

    else:

        VOICE_PROMPT_CACHE[
            voice_name
        ] = voice_prompt


    return voice_prompt


# ============================================================
# PREVIEW
# ============================================================

def preview_voice(
    voice_name
):

    if not voice_name:

        return None


    path = get_voice_file(
        voice_name
    )


    if not path:

        return None


    return path


# ============================================================
# NORMALIZE
# ============================================================

def normalize_audio(
    audio
):

    if torch.is_tensor(
        audio
    ):

        audio = (
            audio
            .detach()
            .float()
            .cpu()
            .numpy()
        )


    audio = np.asarray(
        audio,
        dtype=np.float32
    )


    audio = np.squeeze(
        audio
    )


    if audio.size == 0:

        return audio


    max_amp = np.max(
        np.abs(audio)
    )


    if max_amp > 1.0:

        audio = (
            audio
            /
            max_amp
        )


    return audio


# ============================================================
# ĐẾM TỪ
# ============================================================

def count_text(
    text
):

    text = (
        text
        if text is not None
        else ""
    )


    clean_text = (
        str(text)
        .strip()
    )


    if clean_text:

        words = len(
            re.findall(
                r"\S+",
                clean_text
            )
        )

    else:

        words = 0


    chars = len(
        str(text)
    )


    return (
        f"{words:,} từ"
        f"  •  "
        f"{chars:,} ký tự"
    )


# ============================================================
# CLONE GIỌNG
# ============================================================

def create_custom_voice(
    audio_file,
    voice_name,
    progress=gr.Progress()
):

    global CUSTOM_VOICE_COUNTER


    if not audio_file:

        raise gr.Error(
            "Hãy upload hoặc thu âm "
            "giọng mẫu trước."
        )


    if not os.path.exists(
        audio_file
    ):

        raise gr.Error(
            "Không tìm thấy file âm thanh."
        )


    progress(
        0.10,
        desc="🎤 Đang chuẩn bị..."
    )


    voice_name = (

        voice_name.strip()

        if voice_name

        else "Giọng của tôi"
    )


    if not voice_name:

        voice_name = (
            "Giọng của tôi"
        )


    CUSTOM_VOICE_COUNTER += 1


    display_name = (
        f"🎤 {voice_name} "
        f"#{CUSTOM_VOICE_COUNTER}"
    )


    extension = os.path.splitext(
        audio_file
    )[1]


    if not extension:

        extension = ".wav"


    destination = os.path.join(

        CUSTOM_VOICE_DIR,

        (
            f"custom_voice_"
            f"{CUSTOM_VOICE_COUNTER}"
            f"{extension}"
        )
    )


    shutil.copy2(
        audio_file,
        destination
    )


    progress(
        0.40,
        desc="🧬 Đang clone giọng..."
    )


    try:

        voice_prompt = (
            model.create_voice_clone_prompt(
                ref_audio=destination
            )
        )


    except Exception as e:

        if os.path.exists(
            destination
        ):

            os.remove(
                destination
            )


        raise gr.Error(
            f"Không thể clone giọng:\n{e}"
        )


    CUSTOM_VOICE_FILES[
        display_name
    ] = destination


    CUSTOM_VOICE_PROMPTS[
        display_name
    ] = voice_prompt


    progress(
        1.0,
        desc="✅ Clone hoàn tất"
    )


    return (

        gr.Dropdown(
            choices=get_all_voice_names(),
            value=display_name,
            filterable=False,
            allow_custom_value=False
        ),

        destination,

        (
            f"✅ Đã tạo giọng "
            f"**{display_name}**"
        )
    )


# ============================================================
# TẠO GIỌNG - CHỈ TIẾNG VIỆT
# ============================================================

def generate_voice(
    text,
    voice_name,
    speed,
    pause_duration,
    progress=gr.Progress()
):

    # ========================================================
    # CHECK TEXT
    # ========================================================

    if not text:

        raise gr.Error(
            "Bạn chưa nhập nội dung."
        )


    text = text.strip()


    if not text:

        raise gr.Error(
            "Bạn chưa nhập nội dung."
        )


    # ========================================================
    # CHECK VOICE
    # ========================================================

    if not voice_name:

        raise gr.Error(
            "Bạn chưa chọn giọng."
        )


    if voice_name not in get_all_voice_names():

        raise gr.Error(
            "Giọng không hợp lệ."
        )


    # ========================================================
    # VOICE PROMPT
    # ========================================================

    progress(
        0.05,
        desc="🎤 Đang chuẩn bị giọng..."
    )


    try:

        voice_prompt = (
            get_voice_prompt(
                voice_name
            )
        )


    except Exception as e:

        raise gr.Error(
            f"Lỗi xử lý giọng:\n{e}"
        )


    # ========================================================
    # CHIA ĐOẠN
    # ========================================================

    paragraphs = [

        p.strip()

        for p in re.split(
            r"\n\s*\n",
            text
        )

        if p.strip()
    ]


    if not paragraphs:

        raise gr.Error(
            "Nội dung không hợp lệ."
        )


    total = len(
        paragraphs
    )


    all_audio = []


    # ========================================================
    # GENERATE
    # ========================================================

    for i, paragraph in enumerate(
        paragraphs
    ):


        progress(

            0.10
            +
            0.80
            *
            (
                i
                /
                total
            ),

            desc=(
                f"🎙️ Đang tạo đoạn "
                f"{i + 1}/{total}..."
            )
        )


        try:

            result = model.generate(

                text=paragraph,

                voice_clone_prompt=voice_prompt,

                language="vi",

                speed=float(
                    speed
                ),

                generation_config=GEN_CFG
            )


        except Exception as e:

            raise gr.Error(
                f"Lỗi đoạn "
                f"{i + 1}:\n{e}"
            )


        audio = normalize_audio(
            result[0]
        )


        all_audio.append(
            audio
        )


        # ====================================================
        # PAUSE
        # ====================================================

        if i < total - 1:

            silence_length = int(

                SAMPLING_RATE
                *
                float(
                    pause_duration
                )
            )


            silence = np.zeros(

                silence_length,

                dtype=np.float32
            )


            all_audio.append(
                silence
            )


    # ========================================================
    # CONCAT
    # ========================================================

    progress(
        0.95,
        desc="🎧 Đang ghép audio..."
    )


    final_audio = np.concatenate(
        all_audio
    )


    final_audio = normalize_audio(
        final_audio
    )


    waveform = (

        final_audio
        *
        32767

    ).astype(
        np.int16
    )


    progress(
        1.0,
        desc="✅ Hoàn tất"
    )


    return (
        SAMPLING_RATE,
        waveform
    )


# ============================================================
# DEFAULT VOICE
# ============================================================

DEFAULT_VOICE = (
    "♂ Adam"
)


if DEFAULT_VOICE not in VOICE_DATA:

    DEFAULT_VOICE = (
        list(
            VOICE_DATA.keys()
        )[0]
    )


DEFAULT_PREVIEW = None


try:

    DEFAULT_PREVIEW = (
        get_voice_file(
            DEFAULT_VOICE
        )
    )

except Exception:

    DEFAULT_PREVIEW = None


# ============================================================
# CSS
# ============================================================

CSS = """

/* ==========================================================
   CONTAINER
========================================================== */

.gradio-container {

    max-width: 1040px !important;

    width: 96% !important;

    margin: 0 auto !important;

    padding: 22px !important;
}


/* ==========================================================
   HEADER
========================================================== */

#app_header {

    margin-bottom: 4px !important;
}


#app_header h1 {

    font-size: 30px !important;

    margin-bottom: 2px !important;
}


/* ==========================================================
   TEXTBOX
========================================================== */

#main_text textarea {

    min-height: 270px !important;

    font-size: 17px !important;

    line-height: 1.65 !important;

    padding: 16px !important;
}


/* ==========================================================
   COUNTER
========================================================== */

#text_counter {

    margin-top: -6px !important;

    margin-bottom: 8px !important;
}


#text_counter input {

    border: none !important;

    background: transparent !important;

    box-shadow: none !important;

    padding-left: 2px !important;

    font-size: 14px !important;

    opacity: 0.75 !important;
}


/* ==========================================================
   GENERATE
========================================================== */

#generate_btn button {

    min-height: 58px !important;

    font-size: 18px !important;

    font-weight: 700 !important;
}


/* ==========================================================
   FOOTER
========================================================== */

footer {

    display: none !important;
}

"""


# ============================================================
# THEME
# ============================================================

THEME = gr.themes.Soft(
    primary_hue="indigo"
)


# ============================================================
# UI
# ============================================================

gr.close_all()


with gr.Blocks(
    title="Phong Subi - OMNIVOICE MOD"
) as demo:


    # ========================================================
    # HEADER
    # ========================================================

    gr.Markdown(
        f"""
# 🎙️ Phong Subi - OMNIVOICE MOD

**🇻🇳 {TOTAL_PRESET_VOICES} giọng Tiếng Việt • Clone giọng riêng**
""",
        elem_id="app_header"
    )


    # ========================================================
    # GIỌNG
    # ========================================================

    with gr.Group():


        gr.Markdown(
            "### 🎤 Giọng đọc"
        )


        voice_selector = gr.Dropdown(

            choices=list(
                VOICE_DATA.keys()
            ),

            value=DEFAULT_VOICE,

            label="Chọn giọng",

            interactive=True,

            # =================================================
            # QUAN TRỌNG:
            # KHÔNG CHO GÕ / EDIT
            # =================================================
            filterable=False,

            allow_custom_value=False
        )


        preview_audio = gr.Audio(

            value=DEFAULT_PREVIEW,

            label="🔊 Nghe thử",

            interactive=False,

            autoplay=False
        )


        # ====================================================
        # CLONE
        # ====================================================

        with gr.Accordion(

            "➕ Clone giọng của bạn",

            open=False

        ):


            gr.Markdown(
                """
Upload file âm thanh hoặc thu âm trực tiếp.

**Mẫu tốt:** một người nói, ít tiếng ồn, không nhạc nền, giọng rõ.

Chỉ clone giọng mà bạn có quyền sử dụng.
"""
            )


            custom_voice_name = gr.Textbox(

                label="Tên giọng",

                placeholder=(
                    "Ví dụ: Giọng của tôi"
                )
            )


            custom_audio = gr.Audio(

                sources=[
                    "upload",
                    "microphone"
                ],

                type="filepath",

                label=(
                    "Upload hoặc thu âm "
                    "giọng mẫu"
                )
            )


            clone_button = gr.Button(

                "✨ Tạo giọng clone",

                variant="secondary"
            )


            clone_status = gr.Markdown()


    # ========================================================
    # NỘI DUNG
    # ========================================================

    with gr.Group():


        gr.Markdown(
            "### 📝 Nội dung"
        )


        text_input = gr.Textbox(

            lines=12,

            label="",

            placeholder=(
                "Nhập nội dung Tiếng Việt tại đây...\n\n"
                "Xuống dòng 2 lần nếu muốn nghỉ "
                "giữa các đoạn."
            ),

            elem_id="main_text"
        )


        # ====================================================
        # COUNTER
        # ====================================================

        text_counter = gr.Textbox(

            value="0 từ  •  0 ký tự",

            label="",

            interactive=False,

            container=False,

            elem_id="text_counter"
        )


        # ====================================================
        # SETTINGS
        # ====================================================

        with gr.Row():


            speed_slider = gr.Slider(

                minimum=0.70,

                maximum=1.30,

                value=0.95,

                step=0.05,

                label="⚡ Tốc độ"
            )


            pause_slider = gr.Slider(

                minimum=0,

                maximum=2.0,

                value=0.3,

                step=0.1,

                label="⏸ Nghỉ đoạn (giây)"
            )


        # ====================================================
        # BUTTON
        # ====================================================

        generate_button = gr.Button(

            "🎙️ TẠO GIỌNG NÓI",

            variant="primary",

            elem_id="generate_btn"
        )


    # ========================================================
    # OUTPUT
    # ========================================================

    with gr.Group():


        gr.Markdown(
            "### 🎧 Kết quả"
        )


        output_audio = gr.Audio(

            label="",

            autoplay=False
        )


    # ========================================================
    # EVENTS
    # ========================================================


    # ========================================================
    # ĐẾM TỪ KHI ĐANG GÕ
    # ========================================================

    text_input.input(

        fn=count_text,

        inputs=text_input,

        outputs=text_counter,

        queue=False,

        show_progress="hidden"
    )


    # ========================================================
    # BACKUP ĐẾM TỪ KHI CHANGE
    # ========================================================

    text_input.change(

        fn=count_text,

        inputs=text_input,

        outputs=text_counter,

        queue=False,

        show_progress="hidden"
    )


    # ========================================================
    # PREVIEW GIỌNG
    # ========================================================

    voice_selector.change(

        fn=preview_voice,

        inputs=voice_selector,

        outputs=preview_audio,

        queue=False,

        show_progress="hidden"
    )


    # ========================================================
    # CLONE
    # ========================================================

    clone_button.click(

        fn=create_custom_voice,

        inputs=[
            custom_audio,
            custom_voice_name
        ],

        outputs=[
            voice_selector,
            preview_audio,
            clone_status
        ],

        concurrency_limit=1,

        show_progress="minimal"
    )


    # ========================================================
    # GENERATE
    # ========================================================

    generate_button.click(

        fn=generate_voice,

        inputs=[
            text_input,
            voice_selector,
            speed_slider,
            pause_slider
        ],

        outputs=output_audio,

        concurrency_limit=1,

        show_progress="full"
    )


# ============================================================
# QUEUE
# ============================================================

demo.queue(
    default_concurrency_limit=1
)


# ============================================================
# CLEAR LOG
# ============================================================

clear_output(
    wait=True
)


print(
    "✅ Phong Subi - OMNIVOICE MOD đã sẵn sàng"
)


# ============================================================
# LAUNCH
# ============================================================

demo.launch(

    server_name="0.0.0.0",

    share=True,

    inline=True,

    debug=True,

    theme=THEME,

    css=CSS
)

✅ Phong Subi - OMNIVOICE MOD đã sẵn sàng
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://1c519de0eef68d11ac.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
